### Downloading and Scaling Cifar10

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import random
import os

# --------------------------
# 1. Set ALL random seeds
# --------------------------
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# --------------------------
# 2. Deterministic backend
# --------------------------
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.use_deterministic_algorithms(True)

# --------------------------
# 3. Worker seed function
# --------------------------
def seed_worker(worker_id):
    worker_seed = seed + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(seed)

# --------------------------
# 4. Data transforms
# --------------------------
transform = transforms.Compose([
    transforms.ToTensor()
])

# --------------------------
# 5. Load datasets
# --------------------------
trainset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# --------------------------
# 6. Reproducible DataLoaders
# --------------------------
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    worker_init_fn=seed_worker,
    generator=g
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    worker_init_fn=seed_worker,
    generator=g
)


### Saving the datasets

In [3]:
import torch

# Convert training set to tensors
train_data = torch.stack([img for img, _ in trainset])
train_labels = torch.tensor([label for _, label in trainset])

# Convert test set to tensors
test_data = torch.stack([img for img, _ in testset])
test_labels = torch.tensor([label for _, label in testset])

# Save them
torch.save({
    "images": train_data,
    "labels": train_labels
}, "train_data.pt")

torch.save({
    "images": test_data,
    "labels": test_labels
}, "test_data.pt")

print("Saved: train_data.pt and test_data.pt")


Saved: train_data.pt and test_data.pt


### Training ResNet-18 for CIFAR-10

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision.models import resnet18
import torchvision.transforms as T

import numpy as np
import random
import os

# --------------------------
# 1. Reproducibility setup
# --------------------------
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.use_deterministic_algorithms(True)

def seed_worker(worker_id):
    worker_seed = seed + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# --------------------------
# 2. Load saved datasets
# --------------------------
train_obj = torch.load("train_data.pt")   # {"images": ..., "labels": ...}
test_obj  = torch.load("test_data.pt")

train_images = train_obj["images"]   # [N, 3, 32, 32], float tensor in [0,1]
train_labels = train_obj["labels"]   # [N], long tensor

test_images = test_obj["images"]
test_labels = test_obj["labels"]

print("Train shape:", train_images.shape, train_labels.shape)
print("Test shape:", test_images.shape, test_labels.shape)

# --------------------------
# 3. Augmented Dataset (NO normalization)
# --------------------------
class AugmentedCIFARDataset(Dataset):
    def __init__(self, images, labels, train=True):
        self.images = images
        self.labels = labels
        self.train = train

        self.to_pil = T.ToPILImage()

        if train:
            self.transform = T.Compose([
                T.RandomCrop(32, padding=4),
                T.RandomHorizontalFlip(),
                # Optional extra aug if you want:
                # T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                T.ToTensor(),  # back to tensor in [0,1]
            ])
        else:
            # For test, no augmentation; just ensure tensor type is consistent
            self.transform = T.Compose([
                T.ToTensor(),  # effectively keeps [0,1]
            ])

    def __len__(self):
        return self.images.size(0)

    def __getitem__(self, idx):
        img = self.images[idx]      # [3,32,32] float in [0,1]
        label = self.labels[idx]

        # Convert tensor -> PIL -> transforms -> tensor again
        img = self.to_pil(img)
        img = self.transform(img)

        return img, label

# Use the augmented datasets
train_dataset = AugmentedCIFARDataset(train_images, train_labels, train=True)
test_dataset  = AugmentedCIFARDataset(test_images, test_labels, train=False)

# --------------------------
# 4. DataLoaders
# --------------------------
trainloader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    worker_init_fn=seed_worker,
    generator=g
)

testloader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    worker_init_fn=seed_worker,
    generator=g
)

# --------------------------
# 5. ResNet-18 for CIFAR-10
# --------------------------
model = resnet18(weights=None)

# Adapt for CIFAR-10: smaller input (32x32), 10 classes
# Use 3x3 conv, stride 1, no initial maxpool
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()

# Final FC layer -> 10 classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10)

model = model.to(device)
print(model)

# --------------------------
# 6. Loss, optimizer, scheduler (Step 2)
# --------------------------
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  # still fine without normalization

optimizer = optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4,
)

scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[60, 80],  # drop LR at epochs 60 and 80
    gamma=0.1
)

# --------------------------
# 7. Train / eval loops
# --------------------------
def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for i, (inputs, labels) in enumerate(trainloader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    print(f"Epoch {epoch}: Train loss {epoch_loss:.4f}, acc {epoch_acc:.2f}%")

def evaluate():
    model.eval()
    correct = 0
    total = 0
    test_loss = 0.0

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            test_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    avg_loss = test_loss / total
    acc = 100.0 * correct / total
    print(f"Test: loss {avg_loss:.4f}, acc {acc:.2f}%")
    return avg_loss, acc

# --------------------------
# 8. Run training
# --------------------------
num_epochs = 100
for epoch in range(1, num_epochs + 1):
    train_one_epoch(epoch)
    evaluate()
    scheduler.step()

# --------------------------
# 9. Script & save model
# --------------------------
model = model.to("cpu")
model = torch.jit.script(model)
torch.jit.save(model, "Resnet.pt")
print("Saved model as Resnet.pt")


Using device: cuda
Train shape: torch.Size([50000, 3, 32, 32]) torch.Size([50000])
Test shape: torch.Size([10000, 3, 32, 32]) torch.Size([10000])
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, moment

### Training RF (Class 0 vs. Rest)

In [1]:
import os
import joblib
import numpy as np
import warnings
import torch
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Set seeds for reproducibility
np.random.seed(42)

# ─────────────── PATHS ────────────────────────────────────────────────────
TRAIN_PTH = "train_data.pt"
TEST_PTH  = "test_data.pt"

OUTPUT_MODEL_DIR  = "Models and Data splits"
OUTPUT_MODEL_PATH = os.path.join(OUTPUT_MODEL_DIR, "random_forest_cifar10_class0_vs_rest_unbalanced.pkl")

# Heatmap output paths
COMBINED_HEATMAP_PATH = os.path.join(OUTPUT_MODEL_DIR, "rf_importance_heatmap_combined.png")
CHANNEL_HEATMAP_BASE  = os.path.join(OUTPUT_MODEL_DIR, "rf_importance_heatmap_channel_{}.png")

os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)

# ─────────── Load Data ────────────────────────────────────────────────────
try:
    train_obj = torch.load(TRAIN_PTH)
    test_obj  = torch.load(TEST_PTH)

    X_train = train_obj["images"].numpy()   # [N,3,32,32]
    y_train = train_obj["labels"].numpy()

    X_test  = test_obj["images"].numpy()
    y_test  = test_obj["labels"].numpy()

except Exception as e:
    print(f"Error loading data: {e}")
    raise SystemExit(1)

print("Loaded CIFAR-10 data:")
print(f"  X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"  X_test  shape: {X_test.shape},  y_test  shape: {y_test.shape}")

# ─────────── Create BINARY labels: class 0 vs REST ────────────────────────
# class_0 = 1, rest = 0
y_train_bin = (y_train == 0).astype(int)
y_test_bin  = (y_test == 0).astype(int)

print("\nBinary labels created (1=class0, 0=rest):")
print(f"  Train positives (class 0): {y_train_bin.sum()} / {len(y_train_bin)}")
print(f"  Test  positives (class 0): {y_test_bin.sum()} / {len(y_test_bin)}")

# ─────────── NO BALANCING ─────────────────────────────────────────────────
X_train_final = X_train.astype(np.float32)
y_train_final = y_train_bin

# ─────────── Normalize if needed ──────────────────────────────────────────
if X_train_final.max() > 1.0:
    X_train_final = X_train_final / 255.0
    X_test        = X_test.astype(np.float32) / 255.0
    warnings.warn("Data normalized to [0,1]")
else:
    X_test = X_test.astype(np.float32)
    warnings.warn("Data already in [0,1]")

# ─────────── Flatten (3×32×32 → 3072 features) ────────────────────────────
X_train_flat = X_train_final.reshape(X_train_final.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)

print(f"\nTraining data reshaped to: {X_train_flat.shape}")
print(f"Test data reshaped to:      {X_test_flat.shape}")

# ─────────── Train Random Forest ──────────────────────────────────────────
print("\nTraining Random Forest (UNBALANCED class 0 vs rest)...")

rf_classifier = RandomForestClassifier(
    n_estimators=100,   # as requested
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(X_train_flat, y_train_final)

print("Training complete.")

# ─────────── Evaluate on full test set ────────────────────────────────────
print("\nEvaluating model...")
y_pred = rf_classifier.predict(X_test_flat)
accuracy = accuracy_score(y_test_bin, y_pred)

print(f"Accuracy on test set: {accuracy:.4f}\n")
print("Classification report (1=class0, 0=rest):")
print(classification_report(
    y_test_bin,
    y_pred,
    target_names=["rest", "class_0"]
))

# ─────────── Save Model ───────────────────────────────────────────────────
try:
    joblib.dump(rf_classifier, OUTPUT_MODEL_PATH)
    print(f"Model saved to {OUTPUT_MODEL_PATH}")
except Exception as e:
    print(f"Error saving model: {e}")

# ─────────── Feature Importances + Heatmaps ───────────────────────────────
print("\nComputing feature importance heatmaps...")

feature_importances = rf_classifier.feature_importances_

H, W, C = 32, 32, 3
pixels_per_channel = H * W  # 1024
total_features = C * pixels_per_channel

if feature_importances.shape[0] != total_features:
    raise ValueError(
        f"Expected {total_features} feature importances but got {feature_importances.shape[0]}"
    )

# Reshape into [C, H, W]
importances_img = feature_importances.reshape(C, H, W)

# Combined importance across channels (e.g., sum, could also use np.mean or np.max)
combined_importance = importances_img.sum(axis=0)  # [H, W]

# Normalize for visualization (avoid division by zero)
if combined_importance.max() > 0:
    combined_norm = combined_importance / combined_importance.max()
else:
    combined_norm = combined_importance

# Decide how many "top" pixels to highlight
TOP_K = 768

# Get sorted indices for all features (flattened)
sorted_indices = np.argsort(feature_importances)[::-1]
top_indices = sorted_indices[:TOP_K]

# Convert top feature indices to (channel, row, col)
top_coords = []
for idx in top_indices:
    channel = idx // pixels_per_channel
    rem = idx % pixels_per_channel
    row = rem // W
    col = rem % W
    top_coords.append((channel, row, col))

# ─────────── Print Top K Features (as before) ─────────────────────────────
print(f"\nTop {TOP_K} most important features:")
for rank, (idx, (ch, r, c)) in enumerate(zip(top_indices, top_coords), 1):
    print(
        f"{rank}. Feature {idx} "
        f"(channel {ch}, row {r}, col {c}): "
        f"{feature_importances[idx]:.6f}"
    )

# ─────────── Plot Combined Heatmap + Top Pixels ───────────────────────────
plt.figure(figsize=(6, 6))
plt.imshow(combined_norm, cmap="hot", interpolation="nearest")
plt.title("Random Forest Feature Importance (Combined over RGB)")
plt.colorbar(label="Normalized importance")

# Overlay top features (ignore channel for placement, just use row/col)
for _, (_, row, col) in zip(range(TOP_K), top_coords):
    # small marker at pixel location
    plt.scatter(col + 0.5, row + 0.5, s=40, facecolors='none', edgecolors='cyan', linewidths=1.0)

plt.gca().invert_yaxis()  # so row 0 is at top visually
plt.tight_layout()
plt.savefig(COMBINED_HEATMAP_PATH, dpi=200)
plt.close()

print(f"\nSaved combined importance heatmap to: {COMBINED_HEATMAP_PATH}")

# ─────────── Per-Channel Heatmaps with Top Pixels ─────────────────────────
channel_names = ["Red (0)", "Green (1)", "Blue (2)"]

for ch in range(C):
    ch_importance = importances_img[ch]

    if ch_importance.max() > 0:
        ch_norm = ch_importance / ch_importance.max()
    else:
        ch_norm = ch_importance

    plt.figure(figsize=(6, 6))
    plt.imshow(ch_norm, cmap="hot", interpolation="nearest")
    plt.title(f"Feature Importance - Channel {channel_names[ch]}")
    plt.colorbar(label="Normalized importance")

    # Overlay only top features from this channel
    for _, (channel, row, col) in zip(range(TOP_K), top_coords):
        if channel == ch:
            plt.scatter(col + 0.5, row + 0.5, s=40, facecolors='none',
                        edgecolors='cyan', linewidths=1.0)

    plt.gca().invert_yaxis()
    plt.tight_layout()

    channel_path = CHANNEL_HEATMAP_BASE.format(ch)
    plt.savefig(channel_path, dpi=200)
    plt.close()

    print(f"Saved channel {ch} importance heatmap to: {channel_path}")

print("\nDone. Heatmaps with highlighted top features have been generated.")


Loaded CIFAR-10 data:
  X_train shape: (50000, 3, 32, 32), y_train shape: (50000,)
  X_test  shape: (10000, 3, 32, 32),  y_test  shape: (10000,)

Binary labels created (1=class0, 0=rest):
  Train positives (class 0): 5000 / 50000
  Test  positives (class 0): 1000 / 10000

Training data reshaped to: (50000, 3072)
Test data reshaped to:      (10000, 3072)

Training Random Forest (UNBALANCED class 0 vs rest)...


C:\Users\dyari\AppData\Local\Temp\ipykernel_4668\1730420760.py:65: UserWarning: Data already in [0,1]
  warnings.warn("Data already in [0,1]")


Training complete.

Evaluating model...
Accuracy on test set: 0.9217

Classification report (1=class0, 0=rest):
              precision    recall  f1-score   support

        rest       0.93      0.99      0.96      9000
     class_0       0.81      0.28      0.42      1000

    accuracy                           0.92     10000
   macro avg       0.87      0.64      0.69     10000
weighted avg       0.91      0.92      0.90     10000

Model saved to Models and Data splits\random_forest_cifar10_class0_vs_rest_unbalanced.pkl

Computing feature importance heatmaps...

Top 768 most important features:
1. Feature 2253 (channel 2, row 6, col 13): 0.003182
2. Feature 2252 (channel 2, row 6, col 12): 0.003170
3. Feature 2895 (channel 2, row 26, col 15): 0.002492
4. Feature 2223 (channel 2, row 5, col 15): 0.002239
5. Feature 2257 (channel 2, row 6, col 17): 0.002090
6. Feature 2226 (channel 2, row 5, col 18): 0.002071
7. Feature 2318 (channel 2, row 8, col 14): 0.001992
8. Feature 2859 (channe

### Attack on class 0 (airplanes)  (Class 0 vs. Rest)

In [3]:
import os
import warnings
import numpy as np
import torch
import joblib
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch.nn.functional as F
import time

# ----------------------------------
# 0. Reproducibility
# ----------------------------------
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# ----------------------------------
# 1. PATHS
# ----------------------------------
RESNET_MODEL_PATH = os.path.normpath("Resnet.pt")  # TorchScript ResNet-18 (CIFAR-10)
TEST_PTH = os.path.normpath("test_data.pt")        # {"images": [N,3,32,32], "labels": [N]}

# Random Forest trained on CIFAR-10 pixels (class 0 vs rest, unbalanced)
RF_MODEL_PATH = os.path.normpath(
    "Models and Data splits/random_forest_cifar10_class0_vs_rest_unbalanced.pkl"
)

OUT_DIR = "adversarial_cifar10_resnet_class0_pairs"
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(RF_MODEL_PATH):
    raise FileNotFoundError(f"Random Forest model not found at {RF_MODEL_PATH}")

# ----------------------------------
# 2. HYPER-PARAMETERS
# ----------------------------------
EPSILON_STEP = 0.01
ITERATIONS = 100
TOTAL_EPSILON = 0.5
TOP_K_FEATURES = 768
SAMPLES_PER_CLASS = 100
MAX_L2 = 10000

H, W, C = 32, 32, 3
PIXELS_PER_CHANNEL = H * W
TOTAL_FEATURES = C * PIXELS_PER_CHANNEL  # 3 * 32 * 32 = 3072

# ----------------------------------
# 3. Helper: plotting (for interactive inspection, optional)
# ----------------------------------
def plot_adversarial_example(original_img, adv_img, true_label, adv_label,
                             l2_mag, sample_idx, queries, title_suffix=""):
    def prep(img):
        arr = np.array(img)
        # CHW -> HWC if needed
        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[1:] == (H, W):
            arr = np.transpose(arr, (1, 2, 0))
        if arr.dtype != np.uint8:
            arr = np.clip(arr * 255.0, 0, 255).astype(np.uint8)
        return arr

    orig = prep(original_img)
    adv = prep(adv_img)

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(orig)
    axes[0].set_title(f"Original\nTrue: {true_label}")
    axes[0].axis('off')

    axes[1].imshow(adv)
    axes[1].set_title(f"Adversarial\nPred: {adv_label}")
    axes[1].axis('off')

    fig.suptitle(
        f"Sample {sample_idx} | L2: {l2_mag:.2f} | Queries: {queries} {title_suffix}"
    )
    plt.show()
    plt.close(fig)

# ----------------------------------
# Utility: save before/after + side-by-side
# ----------------------------------
def save_before_after_pair(x0_float_chw, adv_uint8_chw, true_label, adv_label,
                           l2_mag, sample_idx, out_dir):
    """
    x0_float_chw: [3,32,32] float in [0,1]
    adv_uint8_chw: [3,32,32] uint8
    Saves:
      - original_*.png
      - adv_*.png
      - pair_*.png (side-by-side before/after)
    """
    # Original as uint8
    orig_uint8_chw = np.clip(x0_float_chw * 255.0, 0, 255).astype(np.uint8)

    # CHW -> HWC
    orig_hwc = np.transpose(orig_uint8_chw, (1, 2, 0))
    adv_hwc = np.transpose(adv_uint8_chw, (1, 2, 0))

    # Filenames
    base = f"class0_sample{sample_idx}_true{true_label}_adv{adv_label}_mag{l2_mag:.1f}"

    orig_path = os.path.join(out_dir, f"orig_{base}.png")
    adv_path = os.path.join(out_dir, f"adv_{base}.png")
    pair_path = os.path.join(out_dir, f"pair_{base}.png")

    # Save individual images
    Image.fromarray(orig_hwc, mode="RGB").save(orig_path)
    Image.fromarray(adv_hwc, mode="RGB").save(adv_path)

    # Create side-by-side pair image
    pair = np.zeros((H, 2 * W, 3), dtype=np.uint8)
    pair[:, :W, :] = orig_hwc
    pair[:, W:, :] = adv_hwc
    Image.fromarray(pair, mode="RGB").save(pair_path)

    return orig_path, adv_path, pair_path

# ----------------------------------
# 4. Load CIFAR-10 test data
# ----------------------------------
try:
    test_obj = torch.load(TEST_PTH)
    X_samples = test_obj["images"].numpy()   # [N,3,32,32], float in [0,1]
    y_samples = test_obj["labels"].numpy()   # [N]
except FileNotFoundError:
    print(f"Error: Data file not found at {TEST_PTH}. Please check the path.")
    raise SystemExit(1)

print("Loaded CIFAR-10 test data:")
print(f"  X_samples shape: {X_samples.shape}, y_samples shape: {y_samples.shape}")

if X_samples.max() > 1.0:
    X_samples = X_samples.astype(np.float32) / 255.0
    warnings.warn("Data appeared in [0,255]; normalized to [0,1].")

# ----------------------------------
# 5. Load TorchScript ResNet-18
# ----------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    model = torch.jit.load(RESNET_MODEL_PATH, map_location=device)
    model.eval()
    print(f"ResNet TorchScript model loaded from: {RESNET_MODEL_PATH}")
except Exception as e:
    print(f"Error loading TorchScript ResNet model: {e}")
    raise SystemExit(1)

def to_model(x01_batch):
    x = np.array(x01_batch, dtype=np.float32)

    if x.ndim == 3:
        if x.shape == (C, H, W):      # CHW
            x = x[None, ...]
        elif x.shape == (H, W, C):    # HWC
            x = np.transpose(x, (2, 0, 1))[None, ...]
        else:
            raise ValueError(f"Unsupported 3D shape for CIFAR-10: {x.shape}")

    elif x.ndim == 4:
        if x.shape[1:] == (C, H, W):
            pass
        elif x.shape[1:] == (H, W, C):
            x = np.transpose(x, (0, 3, 1, 2))
        else:
            raise ValueError(f"Unsupported 4D shape for CIFAR-10: {x.shape}")

    elif x.ndim == 1 and x.size == TOTAL_FEATURES:
        x = x.reshape(1, C, H, W)
    elif x.ndim == 2 and x.shape[1] == TOTAL_FEATURES:
        x = x.reshape(x.shape[0], C, H, W)
    else:
        raise ValueError(f"Unsupported input shape for to_model: {x.shape}")

    tensor_batch = torch.from_numpy(x).to(device)
    return tensor_batch

def model_query(x_tensor_batch):
    with torch.no_grad():
        return model(x_tensor_batch)

# ----------------------------------
# 6. Load Random Forest & top features
# ----------------------------------
rf_model = None
top_feature_indices = None
print("Loading Random Forest Classifier (CIFAR-10, class0 vs rest)...")
try:
    rf_model = joblib.load(RF_MODEL_PATH)
    feature_importances = rf_model.feature_importances_
    if feature_importances.shape[0] != TOTAL_FEATURES:
        raise ValueError(
            f"Expected {TOTAL_FEATURES} feature importances but got {feature_importances.shape[0]}"
        )
    top_feature_indices = np.argsort(-feature_importances)[:TOP_K_FEATURES]
    print(f"Random Forest loaded. Using top {TOP_K_FEATURES} features.")
except Exception as e:
    print(f"Error loading Random Forest: {e}")
    top_feature_indices = np.arange(TOTAL_FEATURES)

# ----------------------------------
# 7. I-FGSM Attack (3x32x32)
# ----------------------------------
def ifgsm_attack(
    original_image_np,
    true_label,
    model_target,
    epsilon_step,
    iterations,
    total_epsilon_budget,
    features_to_perturb,
    query_count_tracker,
):
    img = np.array(original_image_np, dtype=np.float32)
    if img.ndim == 1 and img.size == TOTAL_FEATURES:
        img = img.reshape(C, H, W)
    elif img.ndim == 3 and img.shape == (H, W, C):
        img = np.transpose(img, (2, 0, 1))
    elif img.ndim == 3 and img.shape == (C, H, W):
        pass
    else:
        raise ValueError(f"Unsupported original image shape: {img.shape}")

    original_image_np = img
    perturbed_image_np = original_image_np.copy()
    final_pred_label = true_label

    for _ in range(iterations):
        image_tensor = to_model(perturbed_image_np)
        image_tensor.requires_grad_(True)

        output = model_target(image_tensor)
        query_count_tracker[0] += 1

        loss = F.cross_entropy(output, torch.tensor([true_label], device=device))
        model_target.zero_grad()
        loss.backward()

        data_grad_sign = torch.sign(image_tensor.grad).cpu().numpy()  # [1,3,32,32]
        gradient_mask = np.zeros_like(data_grad_sign, dtype=np.float32)

        channels = features_to_perturb // PIXELS_PER_CHANNEL
        rem = features_to_perturb % PIXELS_PER_CHANNEL
        rows = rem // W
        cols = rem % W

        valid = (
            (channels >= 0) & (channels < C) &
            (rows >= 0) & (rows < H) &
            (cols >= 0) & (cols < W)
        )
        channels = channels[valid]
        rows = rows[valid]
        cols = cols[valid]

        gradient_mask[0, channels, rows, cols] = data_grad_sign[0, channels, rows, cols]
        gradient_mask_for_update = gradient_mask[0]  # [3,32,32]

        perturbed_image_np += epsilon_step * gradient_mask_for_update

        perturbation = perturbed_image_np - original_image_np
        perturbation = np.clip(perturbation, -total_epsilon_budget, total_epsilon_budget)
        perturbed_image_np = original_image_np + perturbation

        perturbed_image_np = np.clip(perturbed_image_np, 0, 1)

        adv_uint8_temp = np.round(perturbed_image_np * 255).astype(np.uint8)
        adv_float_temp = adv_uint8_temp.astype(np.float32) / 255.0

        with torch.no_grad():
            output_temp = model_target(to_model(adv_float_temp))
            pred_temp = output_temp.argmax(dim=1).item()
            query_count_tracker[0] += 1

        if pred_temp != true_label:
            final_pred_label = pred_temp
            break

    adv_uint8 = np.round(perturbed_image_np * 255).astype(np.uint8)
    adv_float = adv_uint8.astype(np.float32) / 255.0

    with torch.no_grad():
        output_final = model_target(to_model(adv_float))
        final_pred_label = output_final.argmax(dim=1).item()

    l2_norm = np.linalg.norm(
        adv_uint8.astype(np.float32) -
        (original_image_np * 255).astype(np.float32)
    )

    success = (final_pred_label != true_label) and (l2_norm <= MAX_L2)
    return adv_uint8, final_pred_label, l2_norm, success

# ----------------------------------
# 8. Attack Execution – ONLY CLASS 0
# ----------------------------------
print("\n===== Starting Adversarial Attack on CIFAR-10 / ResNet (Class 0 only) =====")
total_trials, succ_total, misclassified = 0, 0, 0
records = []

start_time = time.time()

# Only class 0
class_of_interest = 0
idxs = np.where(y_samples == class_of_interest)[0][:SAMPLES_PER_CLASS]

for rank, idx in enumerate(idxs, 1):
    x0 = X_samples[idx].copy()   # [3,32,32] float
    y0 = int(y_samples[idx])
    query_count_tracker = [0]

    pred0 = model_query(to_model(x0)).argmax(dim=1).item()
    query_count_tracker[0] += 1

    # Save original image even if already misclassified
    orig_uint8_chw = np.clip(x0 * 255.0, 0, 255).astype(np.uint8)
    orig_hwc = np.transpose(orig_uint8_chw, (1, 2, 0))
    orig_name = f"class0_sample{idx}_true{y0}_pred{pred0}_original_only.png"
    Image.fromarray(orig_hwc, mode="RGB").save(os.path.join(OUT_DIR, orig_name))

    if pred0 != y0:
        misclassified += 1
        records.append({
            "sample_idx": idx,
            "true_label": y0,
            "initial_pred": pred0,
            "adv_label": None,
            "success": False,
            "queries": query_count_tracker[0],
            "l2_mag": np.nan,
            "note": "Already misclassified (no attack run)",
        })
        print(
            f"Skipping sample {rank}/{SAMPLES_PER_CLASS} "
            f"(Index {idx}): already misclassified (pred={pred0})"
        )
        continue

    total_trials += 1
    print(
        f"Attacking class 0 sample {rank}/{SAMPLES_PER_CLASS} "
        f"(True: {y0}, Index: {idx})..."
    )

    adv_img_uint8, adv_label, l2_mag, success = ifgsm_attack(
        x0,
        y0,
        model,
        EPSILON_STEP,
        ITERATIONS,
        TOTAL_EPSILON,
        top_feature_indices,
        query_count_tracker,
    )

    # Save before / after / pair for ALL attacked samples
    save_before_after_pair(
        x0,
        adv_img_uint8,
        y0,
        adv_label,
        l2_mag,
        idx,
        OUT_DIR,
    )

    if success:
        succ_total += 1

    records.append({
        "sample_idx": idx,
        "true_label": y0,
        "initial_pred": pred0,
        "adv_label": adv_label,
        "success": success,
        "queries": query_count_tracker[0],
        "l2_mag": l2_mag if success else l2_mag,  # store L2 regardless
        "note": "I-FGSM" + (" successful" if success else " failed"),
    })

    print(
        f"  {'Success' if success else 'Failed'}! "
        f"Adv label: {adv_label}, L2: {l2_mag:.2f}, "
        f"Queries: {query_count_tracker[0]}"
    )

end_time = time.time()
elapsed_time = end_time - start_time
print(f"Time taken to generate adversarial samples: {elapsed_time:.4f} seconds")

# ----------------------------------
# 9. Save Results
# ----------------------------------
df = pd.DataFrame(records)
csv_path = os.path.join(OUT_DIR, "per_sample_stats_class0.csv")
df.to_csv(csv_path, index=False)
print(f"\nStats saved to: {csv_path}")

# ----------------------------------
# 10. Summary
# ----------------------------------
print("\n===== Attack Summary (Class 0) =====")
print(f"Total trials (attacked): {total_trials}")
print(f"Misclassified before attack (skipped): {misclassified}")
if total_trials > 0:
    success_rate = succ_total / total_trials * 100
    print(f"Success rate: {succ_total}/{total_trials} ({success_rate:.1f}%)")
    if succ_total > 0:
        print(
            f"Mean L2 (successful): "
            f"{df[df['success']]['l2_mag'].mean():.2f}"
        )
        print(
            f"Mean queries (successful): "
            f"{df[df['success']]['queries'].mean():.2f}"
        )

# ----------------------------------
# 11. (Optional) Show a few before/after pairs
# ----------------------------------
success_df = df[df["success"]]
if not success_df.empty:
    print("\nShowing successful adversarial examples (from saved pairs)...")
    show_ids = success_df.sample(min(3, len(success_df))).index.tolist()
    for record_idx in show_ids:
        r = df.loc[record_idx]
        # Reload the saved pair
        adv_label = r["adv_label"]
        sample_idx = r["sample_idx"]
        l2_mag = r["l2_mag"]
        base = f"class0_sample{sample_idx}_true{r['true_label']}_adv{adv_label}_mag{l2_mag:.1f}"
        orig_path = os.path.join(OUT_DIR, f"orig_{base}.png")
        adv_path = os.path.join(OUT_DIR, f"adv_{base}.png")
        if os.path.exists(orig_path) and os.path.exists(adv_path):
            orig_img = np.array(Image.open(orig_path))
            adv_img = np.array(Image.open(adv_path))
            plot_adversarial_example(
                orig_img,
                adv_img,
                r["true_label"],
                r["adv_label"],
                r["l2_mag"],
                r["sample_idx"],
                r["queries"],
                "(I-FGSM Class 0)",
            )


Loaded CIFAR-10 test data:
  X_samples shape: (10000, 3, 32, 32), y_samples shape: (10000,)
ResNet TorchScript model loaded from: Resnet.pt
Loading Random Forest Classifier (CIFAR-10, class0 vs rest)...
Random Forest loaded. Using top 768 features.

===== Starting Adversarial Attack on CIFAR-10 / ResNet (Class 0 only) =====
Attacking class 0 sample 1/100 (True: 0, Index: 3)...
  Success! Adv label: 8, L2: 83.14, Queries: 3
Attacking class 0 sample 2/100 (True: 0, Index: 10)...
  Success! Adv label: 9, L2: 341.80, Queries: 29
Attacking class 0 sample 3/100 (True: 0, Index: 21)...
  Success! Adv label: 3, L2: 79.53, Queries: 3
Attacking class 0 sample 4/100 (True: 0, Index: 27)...
  Success! Adv label: 2, L2: 443.45, Queries: 55
Attacking class 0 sample 5/100 (True: 0, Index: 44)...
  Success! Adv label: 2, L2: 876.09, Queries: 121
Attacking class 0 sample 6/100 (True: 0, Index: 52)...
  Success! Adv label: 3, L2: 83.14, Queries: 3
Attacking class 0 sample 7/100 (True: 0, Index: 74)...
 